In [3]:
"""
chunk_and_embed.py — Chunking + Embedding Pipeline for NYC Intelligence

Implements two chunking strategies for the Mayor's Management Report:
  - fixed:         512-token chunks with 50-token overlap
  - section_aware: chunks respect agency section boundaries from Module 07

Usage:
  python chunk_and_embed.py --strategy fixed
  python chunk_and_embed.py --strategy section_aware
  python chunk_and_embed.py --strategy section_aware --source mmr_2024
  python chunk_and_embed.py --mode gutcheck

Prerequisites:
  - chunks_migration.sql has been run against your Neon instance
  - raw_documents table has MMR content (from Episode 07)
  - OPENAI_API_KEY and DATABASE_URL set in .env
"""

import argparse
import hashlib
import json
import os
import re
import sys
from contextlib import contextmanager

import tiktoken
from dotenv import load_dotenv
from openai import OpenAI
import psycopg2
from psycopg2.extras import RealDictCursor

load_dotenv()

True

In [42]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIMS = 1536          # must match the VECTOR() dimension in the table
CHUNK_SIZE = 512               # tokens
CHUNK_OVERLAP = 50             # tokens (10% of 512 — a good default)
BATCH_SIZE = 100               # chunks per embedding API call


In [43]:
# ---------------------------------------------------------------------------
# Database connection
# ---------------------------------------------------------------------------

@contextmanager
def get_db_connection():
    """Context manager for a psycopg2 connection. Closes on exit."""
    conn = psycopg2.connect(os.environ["NEON_DATABASE_URL"])
    try:
        yield conn
    finally:
        conn.close()

In [44]:
# ---------------------------------------------------------------------------
# Tokenizer
# ---------------------------------------------------------------------------

# cl100k_base is the encoding used by text-embedding-3-small and text-embedding-3-large.
# Using the right tokenizer ensures chunk_size token counts match what the API bills.
enc = tiktoken.get_encoding("cl100k_base")

In [45]:
enc

<Encoding 'cl100k_base'>

In [46]:
# ---------------------------------------------------------------------------
# Chunking strategies
# ---------------------------------------------------------------------------

def fixed_chunk(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """
    Split text into fixed-size token chunks with overlap.

    Args:
        text:       Input text to chunk
        chunk_size: Maximum tokens per chunk
        overlap:    Tokens shared between adjacent chunks (prevents boundary loss)

    Returns:
        List of text strings, each at most chunk_size tokens long.

    TODO: Implement this function.
    Hint: Use enc.encode() to tokenize, slice into chunks with stride = chunk_size - overlap,
          then decode each chunk back to text with enc.decode().
    """

    tokens = enc.encode(text)
    chunks = [enc.decode(tokens[i:i + chunk_size]) for i in range(0, len(tokens), chunk_size - overlap)]
    return chunks

In [47]:
text_sample = '''Data-driven law enforcement that deters dangerous driving behavior reduces traffic fatalities and serious injuries. With this in mind, Vision Zero agencies (including NYPD and TLC) continue to focus on enforcing especially hazardous driving violations including speeding, failure-to-yield to pedestrians, signal violations, improper turns, and the use of hand-held devices while driving. In the first four months of Fiscal 2026, six percent more Vision Zero-related moving summonses were issued in total compared to the same period of Fiscal 2025 (from 123,745 to 130,592). This increase was driven by NYPD’s six percent increase (from 119,472 to 126,025) whereas TLC issued one percent fewer (from 4,600 to 4,567). Despite the slight decrease, TLC continued to target unlicensed activity in the first four months of Fiscal 2026, which included illegal operators at airports and cruise terminals. Unlicensed operators pose a risk to public safety, and New Yorkers rely on the drivers and vehicles licensed by TLC to provide safe transportation services. 

The Vision Zero program makes streets safer by removing unregistered and illegal vehicles from City streets. NYPD’s ongoing seizure operations for illegally operated motorcycles, mopeds, and scooters are driving more compliance citywide and therefore reducing the number of illegal devices on the roadways. As such, in the first four months of Fiscal 2026, the number of motorcycles seized decreased eight percent (from 2,483 to 2,289) and the number of mopeds and scooters seized decreased 20 percent (from 6,812 to 5,460) compared to the first four months of Fiscal 2025. Despite the decrease, moped and scooter seizures remain a key priority for NYPD and the level of seizures is still significantly higher than during the first two full fiscal years of reporting (2,773 in Fiscal 2022 and 5,509 in Fiscal 2023). Also contributing to the decrease in seizures is resource reallocation by NYPD to conduct citywide ghost license plate operations starting in Fiscal 2025. In addition to these direct enforcement efforts, NYPD convenes weekly at TrafficStat, where borough police commanders and Vision Zero partners discuss current traffic safety trends, enforcement, education, and engineering. 

## WHAT WE DO 

Established in 1845, the New York City Police Department (NYPD) is responsible for policing an 8.8-million-person city. It performs a wide variety of public safety, law enforcement, traffic management, counterterrorism, and emergency response roles. The NYPD is divided into major bureaus for enforcement, investigations, and administration. It has 78 patrol precincts with patrol officers and detectives covering the entire City. The Department also operates 12 transit districts to police the subway system and its nearly three million daily riders, and nine police service areas (PSAs) to patrol New York City Housing Authority’s public housing developments, which are home to more than 500,000 residents. Additionally, uniformed civilians serve as traffic enforcement agents on the City’s busy streets and highways, as school safety agents, protecting public schools and the nearly one million students who attend them, and as police communications technicians, serving within the 911 emergency radio dispatch center. 

## FOCUS ON EQUITY 

NYPD units are staffed, as always, in accordance with an equitable, need-based allocation of police personnel. Each of the City’s 78 precincts, 12 Transit Bureau districts, and nine Housing Bureau PSAs have unique community and operational needs within their geographic boundaries, including factors such as high-profile locations, transient working and visitor populations, and community and qualityof-life concerns. These considerations, along with crime statistics and the volume of 911 calls requiring a police response, inform the equitable deployment of police resources to address the unique problems and challenges faced by communities within their neighborhoods. 

Additionally, the Department employs a multifaceted deployment strategy that integrates crime reduction, precision policing, advanced technology, and community engagement to address all crime conditions that impact public safety and qualityof-life in New York City. The Department has positioned police officers on streets and subway stations citywide to maximize police utility. These deployments stem from data-driven analysis and community intelligence gathered from patrol officers’ daily interactions, as well as field intelligence officers and Quality of Life Teams (Q-Teams). Q-Teams are composed of officers from across the NYPD who undergo specialized training on how to address non-emergency, quality-of-life concerns in partnership with communities to address matters and safety issues. 

The Department is committed to promoting a fair and inclusive workplace by prioritizing the needs, voices, and perspectives of marginalized employees and communities, while fostering equity through policy and regulation, both in and out of the workplace. The NYPD promotes awareness, education, and outreach efforts to improve the quality of life in the workplace and beyond by fostering cultural understanding. 

The Department will continue to prioritize and adapt its police operations to respond to the City’s most vulnerable communities and address the public safety concerns of everyday New Yorkers. These community-first and precision policing efforts—a data-driven law enforcement strategy that focuses on specific crime hot spots and prolific offenders to reduce crime and community impact—coupled with strategic analysis and oversight, ensure fair and equitable policing and safety. 


'''

text_sample

'Data-driven law enforcement that deters dangerous driving behavior reduces traffic fatalities and serious injuries. With this in mind, Vision Zero agencies (including NYPD and TLC) continue to focus on enforcing especially hazardous driving violations including speeding, failure-to-yield to pedestrians, signal violations, improper turns, and the use of hand-held devices while driving. In the first four months of Fiscal 2026, six percent more Vision Zero-related moving summonses were issued in total compared to the same period of Fiscal 2025 (from 123,745 to 130,592). This increase was driven by NYPD’s six percent increase (from 119,472 to 126,025) whereas TLC issued one percent fewer (from 4,600 to 4,567). Despite the slight decrease, TLC continued to target unlicensed activity in the first four months of Fiscal 2026, which included illegal operators at airports and cruise terminals. Unlicensed operators pose a risk to public safety, and New Yorkers rely on the drivers and vehicles li

In [48]:
chunked_text_sample = fixed_chunk(text_sample)

In [49]:
len(chunked_text_sample)

3

In [50]:
len(chunked_text_sample[0])

2487

In [51]:
len(enc.encode(text_sample))

1067

In [52]:
chunked_text_sample[1]

' DO \n\nEstablished in 1845, the New York City Police Department (NYPD) is responsible for policing an 8.8-million-person city. It performs a wide variety of public safety, law enforcement, traffic management, counterterrorism, and emergency response roles. The NYPD is divided into major bureaus for enforcement, investigations, and administration. It has 78 patrol precincts with patrol officers and detectives covering the entire City. The Department also operates 12 transit districts to police the subway system and its nearly three million daily riders, and nine police service areas (PSAs) to patrol New York City Housing Authority’s public housing developments, which are home to more than 500,000 residents. Additionally, uniformed civilians serve as traffic enforcement agents on the City’s busy streets and highways, as school safety agents, protecting public schools and the nearly one million students who attend them, and as police communications technicians, serving within the 911 em

In [56]:
len(enc.encode(chunked_text_sample[0]))

512